# Convert Llama 3.2 1B Instruct to ExecuTorch (.pte) for the Raspberry Pi 5

**Project:** Bharat AI Hackathon, on-device PCB inspection.

**Who runs this:** a teammate whose Hugging Face account has been approved for `meta-llama/Llama-3.2-1B-Instruct`.

**What it does:** downloads Meta's original weights, converts them into ExecuTorch `.pte` files quantized to
4 bits for the XNNPACK CPU backend, and saves them with the tokenizer files into a Google Drive folder to share.

**Why Colab:** the conversion tool runs on Linux or macOS and needs more memory than the Pi can spare.
ExecuTorch is pinned to **1.6.0** because that is the runtime version already installed on the Pi.

**Runtime:** the default CPU runtime is enough. No GPU is needed. Expect roughly 20 to 40 minutes in total.
Run the cells from top to bottom with **Shift + Enter**.

## Step 1: Install ExecuTorch 1.6.0
The extra index gives the CPU-only build of PyTorch, which is a much smaller download.
pip will print red "dependency conflicts" warnings about packages that came with Colab. They can be ignored,
because this notebook does not use those packages.

In [ ]:
!pip install executorch==1.6.0 torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu

## Step 2: Check the installation
This should print `executorch 1.6.0...` and `torch 2.15.0...`. If it prints an error, stop and send it to Aryan.

In [ ]:
!python -c "import importlib.metadata as m, torch, torchao, pytorch_tokenizers, executorch.extension.llm.export.export_llm; print('executorch', m.version('executorch'), '| torch', torch.__version__, '| export tool found')" 

## Step 3: Sign in to Hugging Face
1. Go to https://huggingface.co/settings/tokens and create a token of type **Read**.
2. Run this cell and paste the token into the box.

The token is used only inside this Colab session to download the gated model.

In [ ]:
from huggingface_hub import login
login()

## Step 4: Download the weights and tokenizer
- `original/consolidated.00.pth` and `original/params.json` are the weights and settings in Meta's own format,
  which is what the ExecuTorch Llama exporter reads.
- The tokenizer files turn text into token numbers and back. The Pi needs them next to the model.
- The licence files travel with the model when it is shared.

If this fails with a 403 error, the account is not approved for the model yet.

In [ ]:
from huggingface_hub import hf_hub_download

REPO = "meta-llama/Llama-3.2-1B-Instruct"
SRC = "/content/llama"

required = ["original/consolidated.00.pth", "original/params.json", "original/tokenizer.model",
            "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"]
optional = ["LICENSE.txt", "USE_POLICY.md"]

for name in required:
    print("downloaded:", hf_hub_download(REPO, name, local_dir=SRC))
for name in optional:
    try:
        print("downloaded:", hf_hub_download(REPO, name, local_dir=SRC))
    except Exception as e:
        print("skipped", name, "-", type(e).__name__)

## Step 5: Write the two export settings files
We export two versions, so the Pi has a fallback without a second round trip.

**Main** follows the ExecuTorch documentation for Llama 3.2:
- `use_kv_cache`: the model remembers earlier tokens instead of re-reading the whole prompt for every new token.
- `use_sdpa_with_kv_cache`: a faster attention operator (needs ExecuTorch's LLM custom operators at run time).
- `qmode: 8da4w`: weights stored in 4 bits, activations in 8 bits. This cuts the model to roughly a quarter of its size.
- `embedding_quantize: 4,32`: the word-lookup table is also stored in 4 bits.
- `xnnpack`: hand the heavy maths to XNNPACK, the optimised Arm CPU backend.
- `max_seq_length` / `max_context_length: 1024`: the default is 128 tokens, which is too short for our prompt plus the answer.
- `metadata`: the token numbers that mark "start of text" and "end of answer", so generation knows when to stop.

**Backup** leaves out the custom attention operator and the embedding quantization. It is larger and may be
slower, but it only needs the standard operators plus XNNPACK.

In [ ]:
MAIN_YAML = """base:
  model_class: llama3_2
  checkpoint: /content/llama/original/consolidated.00.pth
  params: /content/llama/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
export:
  max_seq_length: 1024
  max_context_length: 1024
  output_name: /content/llama_out/llama3_2_1b_main.pte
quantization:
  qmode: 8da4w
  embedding_quantize: 4,32
backend:
  xnnpack:
    enabled: True
    extended_ops: True
"""

BACKUP_YAML = """base:
  model_class: llama3_2
  checkpoint: /content/llama/original/consolidated.00.pth
  params: /content/llama/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
export:
  max_seq_length: 1024
  max_context_length: 1024
  output_name: /content/llama_out/llama3_2_1b_backup.pte
quantization:
  qmode: 8da4w
backend:
  xnnpack:
    enabled: True
    extended_ops: True
"""

import os
os.makedirs("/content/llama_out", exist_ok=True)
with open("/content/main.yaml", "w") as f:
    f.write(MAIN_YAML)
with open("/content/backup.yaml", "w") as f:
    f.write(BACKUP_YAML)
print("settings files written")

## Step 6: Export the main model
This is the slow step (several minutes, possibly 15 or more on Colab's CPU). It prints a lot of log lines.

If the output ends with `Killed`, or Colab says the session crashed after using all available RAM, the export
ran out of memory. In that case skip to Step 7 and report it; the same command also works on a Mac or Linux
machine with Python 3.11 to 3.13 after `pip install executorch==1.6.0`.

In [ ]:
!cd /content && python -m executorch.extension.llm.export.export_llm --config /content/main.yaml && ls -lh /content/llama_out

## Step 7: Export the backup model

In [ ]:
!cd /content && python -m executorch.extension.llm.export.export_llm --config /content/backup.yaml && ls -lh /content/llama_out

## Step 8: Copy everything to Google Drive
Colab will ask for permission to access your Drive. The files go into a new folder called `llama_pi_export`
(about 3 GB in total).

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")
dest = "/content/drive/MyDrive/llama_pi_export"
os.makedirs(dest, exist_ok=True)

files = [
    "/content/llama_out/llama3_2_1b_main.pte",
    "/content/llama_out/llama3_2_1b_backup.pte",
    SRC + "/original/tokenizer.model",
    SRC + "/tokenizer.json",
    SRC + "/tokenizer_config.json",
    SRC + "/special_tokens_map.json",
    SRC + "/LICENSE.txt",
    SRC + "/USE_POLICY.md",
]
for path in files:
    if os.path.exists(path):
        shutil.copy(path, dest)
        print(f"copied  {os.path.basename(path):32s} {os.path.getsize(path) / 1e6:8.1f} MB")
    else:
        print("MISSING", path)
drive.flush_and_unmount()
print("\nDone. Folder in your Drive: llama_pi_export")

## Step 9: Share the folder
In Google Drive, right-click the `llama_pi_export` folder, choose **Share**, and add Aryan's Google account
(or set "Anyone with the link" to Viewer and send him the link).

Please also tell him:
- whether Steps 6 and 7 both finished, and
- the file sizes that Step 8 printed.